In [4]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas12-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# Membaca dataset yang dibuat pada cell persiapan modul
df_tugas = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)
df_tugas.printSchema()
df_tugas.show(5)
print("Jumlah data:", df_tugas.count())
df_tugas.groupBy("ulasan_positif").count().show()

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- waktu_tunggu_menit: integer (nullable = true)
 |-- ulasan_positif: integer (nullable = true)

+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
+--------------+----------+-----------+---+------------------+--------------+
only showing top 5 rows

Jumlah data

In [2]:
# A. Membangun ML Pipeline

idx_menu   = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu  = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

ohe = OneHotEncoder(inputCols=["menu_idx","cabang_idx","waktu_idx"],
                    outputCols=["menu_vec","cabang_vec","waktu_vec"])

assembler = VectorAssembler(
    inputCols=["menu_vec","cabang_vec","waktu_vec","qty","waktu_tunggu_menit"],
    outputCol="features")

lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

pipeline_lr = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, lr])

print("tahap pipeline berhasil didefinisikan.")

tahap pipeline berhasil didefinisikan.


In [10]:
# B. Pelatihan & Evaluasi

from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

train, test = df_tugas.randomSplit([0.8, 0.2], seed=77)
print(f"Data training: {train.count()} | Data testing: {test.count()}")

model_lr = pipeline_lr.fit(train)
pred_lr = model_lr.transform(test)

ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
ev_f1  = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")

acc_lr = ev_acc.evaluate(pred_lr)
f1_lr  = ev_f1.evaluate(pred_lr)
auc_lr = ev_auc.evaluate(pred_lr)

print(" Evaluasi Logistic Regression ")
print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

print("\n Confusion Matrix Logistic Regression ")
pred_lr.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

Data training: 2411 | Data testing: 589
 Evaluasi Logistic Regression 
Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650

 Confusion Matrix Logistic Regression 
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



In [11]:
# C. Model Pembanding

rf = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30, seed=77)
pipeline_rf = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, rf])

model_rf = pipeline_rf.fit(train)
pred_rf = model_rf.transform(test)

acc_rf = ev_acc.evaluate(pred_rf)
f1_rf  = ev_f1.evaluate(pred_rf)
auc_rf = ev_auc.evaluate(pred_rf)

print(" Evaluasi Random Forest ")
print(f"Accuracy : {acc_rf:.4f}")
print(f"F1-score : {f1_rf:.4f}")
print(f"AUC-ROC  : {auc_rf:.4f}")

print("\n Perbandingan ")
print(f"Logistic Regression -> Acc: {acc_lr:.4f} | F1: {f1_lr:.4f} | AUC: {auc_lr:.4f}")
print(f"Random Forest       -> Acc: {acc_rf:.4f} | F1: {f1_rf:.4f} | AUC: {auc_rf:.4f}")

 Evaluasi Random Forest 
Accuracy : 0.7963
F1-score : 0.7959
AUC-ROC  : 0.8455

 Perbandingan 
Logistic Regression -> Acc: 0.7929 | F1: 0.7926 | AUC: 0.8650
Random Forest       -> Acc: 0.7963 | F1: 0.7959 | AUC: 0.8455


Berdasarkan hasil evaluasi, Random Forest memiliki Accuracy (0,7963) dan F1-score (0,7959) yang lebih tinggi dibandingkan Logistic Regression. Namun, Logistic Regression memiliki AUC-ROC (0,8650) lebih tinggi, sehingga lebih baik dalam membedakan kelas positif dan negatif.

In [14]:
# D. Analisis Kesalahan Model

auc_lr, auc_rf = ev_auc.evaluate(pred_lr), ev_auc.evaluate(pred_rf)
pred_terbaik = pred_lr if auc_lr >= auc_rf else pred_rf

salah = pred_terbaik.filter(col("ulasan_positif") != col("prediction"))
salah.select("nama_menu","cabang","waktu_pesan","qty","waktu_tunggu_menit", "ulasan_positif","prediction","probability").show(10, truncate=False)

+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.6253879175414343,0.3746120824585657] |
|Americano|Magelang|Siang      |2  |22                |1             |0.0       |[0.9113822566149014,0.08861774338509865]|
|Americano|Magelang|Sore       |1  |7                 |0             |1.0       |[0.23209137939008387,0.7679086206099162]|
|Americano|Magelang|Sore       |2  |11                |0             |1.0       |[0.42615454016905935,0.5738454598309406]|
|Americano|Magelang|Sore       |3  |5                 |0             |1.0       |[0.16534002323316696,0.834659976766833] |
|Americano|Magel

Dari 10 baris data yang salah diprediksi, nilai waktu_tunggu_menit tersebar antara 2 hingga 22 menit sehingga tidak terpusat pada satu ambang batas tertentu. Namun, terlihat pola kesalahan berupa false positive pada waktu tunggu 2–11 menit dan false negative pada waktu tunggu 13–22 menit. Kesalahan ini kemungkinan dipengaruhi oleh kombinasi beberapa fitur dan variasi data, bukan hanya waktu tunggu. Selain itu, seluruh data yang ditampilkan merupakan menu Americano sehingga belum dapat disimpulkan bahwa menu atau cabang tertentu menjadi penyebab kesalahan prediksi.

# E. Kesimpulan Bisnis
Berdasarkan hasil pengujian terhadap 3.000 data ulasan pelanggan, model Logistic Regression memperoleh accuracy sebesar 79,29%, F1-score sebesar 79,26%, dan AUC-ROC sebesar 0,8650. Model ini dipilih sebagai model terbaik dibandingkan Random Forest karena memiliki nilai AUC-ROC yang lebih tinggi dalam membedakan ulasan positif dan negatif. Berdasarkan analisis kesalahan pada Bagian D, kesalahan prediksi terjadi pada berbagai rentang waktu tunggu, yaitu 2 hingga 22 menit, dengan pola false positive pada waktu tunggu 2–11 menit dan false negative pada 13–22 menit. Hal ini menunjukkan bahwa waktu tunggu perlu diperhatikan dalam evaluasi pelayanan, meskipun hasil tersebut belum cukup untuk membuktikan bahwa waktu tunggu merupakan faktor yang paling berpengaruh terhadap ulasan pelanggan. Oleh karena itu, manajemen Kopi Nusantara disarankan untuk menetapkan target waktu penyajian, memantau pesanan yang mengalami keterlambatan, serta mengevaluasi pembagian tugas barista terutama pada jam sibuk. Selain itu, manajemen dapat melakukan evaluasi kualitas produk dan pelayanan di setiap cabang untuk mengetahui aspek yang perlu ditingkatkan. Hasil prediksi model dapat dimanfaatkan sebagai alat bantu dalam mengidentifikasi kecenderungan ulasan negatif dan menentukan langkah perbaikan pelayanan. Dengan demikian, penerapan Machine Learning diharapkan dapat membantu manajemen mengambil keputusan berdasarkan data serta meningkatkan kualitas pelayanan dan kepuasan pelanggan.

In [16]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
